In [2]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests

RAW = Path("../data/raw/censo_locales")
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED = Path("../data/processed")

URL = ("https://datos.madrid.es/dataset/200085-0-censo-locales/resource/"
       "200085-5-censo-locales/download/200085-5-censo-locales.csv")
ARCHIVO = RAW / "actividades.csv"

# Solo descarga si no lo tenemos ya (son ~120 MB)
if not ARCHIVO.exists():
    with requests.get(URL, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(ARCHIVO, "wb") as f:
            for trozo in r.iter_content(chunk_size=1024 * 1024):
                f.write(trozo)

print(f"{ARCHIVO.stat().st_size / 1e6:.0f} MB")

125 MB


In [3]:
# Los CSV del Ayuntamiento suelen ir separados por ";" y a veces en latin-1 en vez de utf-8
for codificacion in ["utf-8", "latin-1"]:
    try:
        censo = pd.read_csv(ARCHIVO, sep=";", dtype=str, encoding=codificacion)
        break
    except UnicodeDecodeError:
        continue
censo.columns = censo.columns.str.strip()
print("Codificación:", codificacion)
print("Filas:", len(censo), "· locales distintos:", censo["id_local"].nunique())
print("Fecha de carga:", censo["fx_carga"].unique()[:3])

print("\n--- Situación de los locales ---")
print(censo["desc_situacion_local"].value_counts())

abiertos = censo[censo["desc_situacion_local"].str.strip().str.lower() == "abierto"]
print("\n--- Secciones de actividad (abiertos) ---")
print(abiertos["desc_seccion"].value_counts().head(15))

print("\n--- Los 60 epígrafes más frecuentes (abiertos) ---")
print(abiertos["desc_epigrafe"].value_counts().head(60).to_string())

print("\n--- Rótulo (nombre del cartel) ---")
print(f"Vacío en el {abiertos['rotulo'].isna().mean():.0%} de los locales abiertos")
print(abiertos["rotulo"].dropna().sample(10, random_state=1).to_string())

print("\n--- Coordenadas ---")
print(abiertos[["coordenada_x_local", "coordenada_y_local"]].head(5))

Codificación: utf-8
Filas: 225966 · locales distintos: 203838
Fecha de carga: <ArrowStringArray>
['27/11/2026']
Length: 1, dtype: str

--- Situación de los locales ---
desc_situacion_local
Abierto               159875
Cerrado                40637
Baja                   12549
Uso vivienda            8495
Baja Reunificación      4410
Name: count, dtype: int64

--- Secciones de actividad (abiertos) ---
desc_seccion
COMERCIO AL POR MAYOR Y AL POR MENOR; REPARACIÓN DE VEHÍCULOS DE MOTOR Y MOTOCICLETAS    45887
HOSTELERÍA                                                                               28011
OTROS SERVICIOS                                                                          14864
ACTIVIDADES ADMINISTRATIVAS Y SERVICIOS AUXLIARES                                         9720
ACTIVIDADES PROFESIONALES, CIENTÍFICAS Y TÉCNICAS                                         9370
ACTIVIDADES SANITARIAS Y DE SERVICIOS SOCIALES                                            8514
EDUCACIÓN     

In [4]:
# Lista completa de actividades de los locales abiertos, con cuántos hay de cada una
epigrafes = (
    abiertos.groupby(["desc_seccion", "desc_division", "desc_epigrafe"])
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)
epigrafes.to_csv(RAW / "epigrafes_abiertos.csv", index=False, encoding="utf-8-sig")
print(len(epigrafes), "actividades distintas")
epigrafes.head(20)

450 actividades distintas


,desc_seccion,desc_division,desc_epigrafe,n
0,ACTIVIDADES ADMINISTRATIVAS Y SERVICIOS AUXLIARES,ACTIVIDADES ADMINISTRATIVAS DE OFICINA Y OTRAS...,ACTIVIDADES ADMINISTRATIVAS Y AUXILIARES DE OF...,7777
291,HOSTELERÍA,SERVICIOS DE ALOJAMIENTO,VIVIENDAS TURÍSTICAS,5286
420,OTROS SERVICIOS,OTROS SERVICIOS PERSONALES,SERVICIO DE PELUQUERIA,5219
296,HOSTELERÍA,SERVICIOS DE COMIDAS Y BEBIDAS,BAR RESTAURANTE,4561
293,HOSTELERÍA,SERVICIOS DE COMIDAS Y BEBIDAS,BAR CON COCINA,3933
210,COMERCIO AL POR MAYOR Y AL POR MENOR; REPARACI...,"COMERCIO AL POR MENOR, EXCEPTO DE VEHÍCULOS DE...",COMERCIO AL POR MENOR DE PRENDAS DE VESTIR EN ...,3576
299,HOSTELERÍA,SERVICIOS DE COMIDAS Y BEBIDAS,CAFETERIA,3227
280,EDUCACIÓN,EDUCACIÓN,"ENSEÑANZA NO REGLADA (DEPORTIVA Y RECREATIVA, ...",2955
305,HOSTELERÍA,SERVICIOS DE COMIDAS Y BEBIDAS,RESTAURANTE,2820
409,OTROS SERVICIOS,OTROS SERVICIOS PERSONALES,CENTRO DE ESTETICA,2698


In [5]:
import unicodedata

def normalizar(texto):
    """Mayúsculas y sin tildes, para comparar nombres escritos de forma distinta."""
    texto = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    return texto.upper().strip()

# Coordenadas a número y fuera las que vienen a 0 (locales sin ubicación)
for col in ["coordenada_x_local", "coordenada_y_local"]:
    abiertos[col] = pd.to_numeric(abiertos[col].str.replace(",", "."), errors="coerce")
con_coords = abiertos[(abiertos["coordenada_x_local"] > 0) & (abiertos["coordenada_y_local"] > 0)].copy()
print(f"Con coordenadas: {len(con_coords)} de {len(abiertos)} ({len(con_coords) / len(abiertos):.1%})")

# Suponemos que están en EPSG:25830 (metros, como nuestros barrios) y lo comprobamos:
# si acertamos, cada local debe caer dentro del barrio que dice el propio censo
barrios = gpd.read_parquet(PROCESSED / "barrios.parquet")[["barrio", "geometry"]].to_crs(25830)
puntos = gpd.GeoDataFrame(
    con_coords,
    geometry=gpd.points_from_xy(con_coords["coordenada_x_local"], con_coords["coordenada_y_local"]),
    crs=25830,
)
cruce = gpd.sjoin(puntos, barrios, predicate="within", how="left")
coinciden = (cruce["barrio"].map(normalizar) == cruce["desc_barrio_local"].map(normalizar)).mean()
print(f"Locales que caen en el barrio que dice el censo: {coinciden:.1%}")

Con coordenadas: 125537 de 159875 (78.5%)
Locales que caen en el barrio que dice el censo: 90.0%


In [6]:
# Reglas: (categoría, expresión regular sobre el epígrafe). Se aplican EN ORDEN y gana la primera.
# Las discotecas van antes que los bares, y "BAR RESTAURANTE" cuenta como restaurante.
REGLAS = [
    ("discotecas",       r"DISCOTECA|SALAS? DE FIESTA|SALAS? DE BAILE|BAR ESPECIAL|\bPUB\b|KARAOKE"),
    ("restaurantes",     r"RESTAURANTE|PIZZERIA|HAMBURGUESERIA|COMIDAS PARA LLEVAR|ASADOR"),
    ("bares_cafeterias", r"\bBAR\b|BARES|CAFETERIA|TABERNA|CERVECERIA|CHOCOLATERIA"),
    ("supermercados",    r"SUPERMERCADO|HIPERMERCADO|AUTOSERVICIO|NO ESPECIALIZADOS.*ALIMENT"),
    ("farmacias",        r"FARMACIA"),
    ("gimnasios",        r"GIMNASIO|FITNESS"),
    ("veterinarios",     r"VETERINARI"),
]

con_coords["epigrafe_norm"] = con_coords["desc_epigrafe"].map(normalizar)
con_coords["categoria"] = None
for categoria, patron in REGLAS:
    sin_asignar = con_coords["categoria"].isna()
    encaja = con_coords["epigrafe_norm"].str.contains(patron, regex=True)
    con_coords.loc[sin_asignar & encaja, "categoria"] = categoria

print("--- Locales por categoría ---")
print(con_coords["categoria"].value_counts())

print("\n--- Qué actividades ha cogido cada regla ---")
revision = (
    con_coords.dropna(subset=["categoria"])
    .groupby(["categoria", "desc_epigrafe"]).size()
    .reset_index(name="n")
    .sort_values(["categoria", "n"], ascending=[True, False])
)
print(revision.to_string(index=False))

print("\n--- Hostelería que se ha quedado SIN categoría ---")
fuera = con_coords[con_coords["categoria"].isna() & (con_coords["desc_seccion"].map(normalizar) == "HOSTELERIA")]
print(fuera["desc_epigrafe"].value_counts().to_string())

--- Locales por categoría ---
categoria
bares_cafeterias    8263
restaurantes        8019
supermercados       2535
farmacias           1846
discotecas          1114
gimnasios            935
veterinarios         359
Name: count, dtype: int64

--- Qué actividades ha cogido cada regla ---
       categoria                                                                                                                                 desc_epigrafe    n
bares_cafeterias                                                                                                                                BAR CON COCINA 3743
bares_cafeterias                                                                                                                                     CAFETERIA 2844
bares_cafeterias                                                                                                                                BAR SIN COCINA 1457
bares_cafeterias                                         

In [7]:
# 1) Probamos también suponiendo que las coordenadas están en ED50 (EPSG:23030)
puntos_ed50 = puntos.set_crs(23030, allow_override=True).to_crs(25830)
cruce_ed50 = gpd.sjoin(puntos_ed50, barrios, predicate="within", how="left")
coinciden_ed50 = (cruce_ed50["barrio"].map(normalizar) == cruce_ed50["desc_barrio_local"].map(normalizar)).mean()
print(f"Suponiendo EPSG:25830: {coinciden:.1%}  ·  suponiendo ED50: {coinciden_ed50:.1%}")

# 2) ¿Qué pares de nombres no coinciden? Si son el mismo barrio escrito distinto, no es problema
no_coinciden = cruce[cruce["barrio"].map(normalizar) != cruce["desc_barrio_local"].map(normalizar)]
print("\n--- Pares que no coinciden (nuestro barrio / barrio del censo) ---")
print(no_coinciden.groupby(["barrio", "desc_barrio_local"]).size().sort_values(ascending=False).head(25).to_string())

Suponiendo EPSG:25830: 90.0%  ·  suponiendo ED50: 66.0%

--- Pares que no coinciden (nuestro barrio / barrio del censo) ---
barrio                                           desc_barrio_local   
Villaverde Alto - Casco Histórico de Villaverde  SAN ANDRES              2798
Casco Histórico de Vallecas                      CASCO H.VALLECAS        2754
Pilar                                            EL PILAR                1666
Casco Histórico de Vicálvaro                     CASCO H.VICALVARO       1144
Águilas                                          LAS AGUILAS             1049
Peñagrande                                       PEÑA GRANDE              947
La Concepción                                    CONCEPCION               937
Ángeles                                          LOS ANGELES              756
Casco Histórico de Barajas                       CASCO H.BARAJAS          418
Valdefuentes                                     TIMON                     22
Timón                     

In [8]:
# Un punto por local y categoría (un local con 2 actividades de la misma categoría cuenta una vez)
locales = (
    puntos[con_coords["categoria"].notna().values]
    .assign(categoria=con_coords.loc[con_coords["categoria"].notna(), "categoria"].values)
    .drop_duplicates(["id_local", "categoria"])
)
locales["nombre"] = locales["rotulo"].str.strip().str.title()
print(f"{len(locales)} locales del censo en nuestras categorías")
print(f"Sin rótulo (nombre): {locales['nombre'].isna().mean():.0%}")

osm = gpd.read_parquet(PROCESSED / "pois.parquet")
CATEGORIAS = list(dict.fromkeys(c for c, _ in REGLAS))

filas = []
for cat in CATEGORIAS:
    c = locales[locales["categoria"] == cat]
    o = osm[osm["categoria"] == cat]
    if c.empty or o.empty:
        continue
    # ¿Qué % de cada fuente tiene un sitio de la otra fuente a menos de 50 m?
    o_en_censo = gpd.sjoin_nearest(o[["geometry"]], c[["geometry"]], max_distance=50, how="inner")
    c_en_osm = gpd.sjoin_nearest(c[["geometry"]], o[["geometry"]], max_distance=50, how="inner")
    filas.append({
        "categoria": cat,
        "censo": len(c),
        "osm": len(o),
        "veces_mas_en_censo": round(len(c) / len(o), 1),
        "% de OSM que está en el censo": f"{o_en_censo.index.nunique() / len(o):.0%}",
        "% del censo que está en OSM": f"{c_en_osm.index.nunique() / len(c):.0%}",
    })

comparacion = pd.DataFrame(filas)
comparacion

22486 locales del censo en nuestras categorías
Sin rótulo (nombre): 0%


,categoria,censo,osm,veces_mas_en_censo,% de OSM que está en el censo,% del censo que está en OSM
0,discotecas,1082,155,7.0,82%,17%
1,restaurantes,7818,6053,1.3,87%,73%
2,bares_cafeterias,7941,3901,2.0,86%,54%
3,supermercados,2513,882,2.8,75%,33%
4,farmacias,1846,1854,1.0,92%,93%
5,gimnasios,935,327,2.9,54%,19%
6,veterinarios,351,170,2.1,82%,40%


In [9]:
URL_TERRAZAS = ("https://datos.madrid.es/dataset/200085-0-censo-locales/resource/"
                "200085-6-censo-locales/download/200085-6-censo-locales.csv")
ARCHIVO_TERRAZAS = RAW / "terrazas.csv"
if not ARCHIVO_TERRAZAS.exists():
    r = requests.get(URL_TERRAZAS, timeout=300)
    r.raise_for_status()
    ARCHIVO_TERRAZAS.write_bytes(r.content)

terrazas = pd.read_csv(ARCHIVO_TERRAZAS, sep=";", dtype=str, encoding=codificacion)
terrazas.columns = terrazas.columns.str.strip()
print(len(terrazas), "terrazas ·", terrazas["id_local"].nunique(), "locales distintos")

print("\n--- Situación de la terraza ---")
print(terrazas["desc_situacion_terraza"].value_counts(dropna=False))
print("\n--- Periodo ---")
print(terrazas["desc_periodo_terraza"].value_counts(dropna=False))

# ¿Cuántos de nuestros bares, restaurantes y discotecas tienen terraza?
con_terraza = set(terrazas["id_local"].str.strip())
locales["terraza"] = locales["id_local"].str.strip().isin(con_terraza)
print("\n--- Locales con terraza por categoría ---")
print(locales.groupby("categoria")["terraza"].agg(["sum", "mean"]).round(2))

6631 terrazas · 6471 locales distintos

--- Situación de la terraza ---
desc_situacion_terraza
Abierta                6579
Suspension temporal      52
Name: count, dtype: int64

--- Periodo ---
desc_periodo_terraza
Anual         5629
Estacional    1002
Name: count, dtype: int64

--- Locales con terraza por categoría ---
                   sum  mean
categoria                   
bares_cafeterias  3167  0.40
discotecas          86  0.08
farmacias            4  0.00
gimnasios            9  0.01
restaurantes      2912  0.37
supermercados       29  0.01
veterinarios         2  0.01


In [10]:
# 1) Solo terrazas abiertas, y solo en hostelería
terrazas_ok = terrazas[terrazas["desc_situacion_terraza"].str.strip() == "Abierta"]
ids_terraza = set(terrazas_ok["id_local"].str.strip())
locales["terraza"] = (
    locales["id_local"].str.strip().isin(ids_terraza)
    & locales["categoria"].isin(["bares_cafeterias", "restaurantes", "discotecas"])
)

# 2) El censo con el mismo formato que pois.parquet (id, name, categoria, geometry)
censo_pois = gpd.GeoDataFrame(
    {
        "id": "censo_" + locales["id_local"].str.strip(),
        "name": locales["nombre"],
        "categoria": locales["categoria"],
        "fuente": "censo",
    },
    geometry=locales.geometry.values, crs=25830,
)
# Categoría nueva: bares y restaurantes con terraza (son los mismos puntos, otra etiqueta)
terrazas_pois = censo_pois[
    locales["terraza"].values & locales["categoria"].isin(["bares_cafeterias", "restaurantes"]).values
].assign(categoria="terrazas")

# 3) De OpenStreetMap: todo lo de las categorías que el censo no cubre...
osm = osm.assign(id="osm_" + osm["id"].astype(str), fuente="osm")
partes = [osm[~osm["categoria"].isin(CATEGORIAS)]]
# ...y, en las que sí cubre, solo los sitios que no tienen un local del censo a menos de 50 m
for cat in CATEGORIAS:
    o = osm[osm["categoria"] == cat]
    c = censo_pois[censo_pois["categoria"] == cat]
    repetidos = gpd.sjoin_nearest(o[["geometry"]], c[["geometry"]], max_distance=50, how="inner").index
    partes.append(o[~o.index.isin(repetidos)])

combinado = gpd.GeoDataFrame(
    pd.concat(partes + [censo_pois, terrazas_pois], ignore_index=True), crs=25830
)
combinado.to_parquet(PROCESSED / "pois_combinados.parquet")

print(f"Antes (solo OSM): {len(osm)} · ahora (OSM + censo): {len(combinado)}")
resumen = combinado.pivot_table(index="categoria", columns="fuente", aggfunc="size", fill_value=0)
resumen["total"] = resumen.sum(axis=1)
resumen.loc[CATEGORIAS + ["terrazas"]]

Antes (solo OSM): 27402 · ahora (OSM + censo): 44468


fuente,censo,osm,total
categoria,,,
discotecas,1082,28,1110
restaurantes,7818,771,8589
bares_cafeterias,7941,546,8487
supermercados,2513,221,2734
farmacias,1846,148,1994
gimnasios,935,150,1085
veterinarios,351,30,381
terrazas,6028,0,6028
